# Lab 8: Multi-Agent Swarm with Bedrock and Strands SDK

## Introduction

In this lab, you'll learn how to create a multi-agent swarm system using Amazon Bedrock and the Strands SDK. Building on the concepts from Lab 7, we'll implement a collaborative agent system where multiple specialized agents work together to solve complex tasks.

By the end of this lab, you'll understand:
- How to create specialized agents with different roles and capabilities
- How to implement agent coordination and handoff mechanisms
- How to use the Strands SDK Swarm pattern for autonomous collaboration
- How to leverage Amazon Bedrock models for multi-agent systems
- Best practices for multi-agent orchestration and task distribution

Let's get started!

## 1. Setup and Installation

First, let's install the necessary libraries:

In [ ]:
# Install required packages using UV
!uv add --quiet strands-agents strands-agents-tools boto3 python-dotenv

Next, let's import the required libraries and set up our environment:

In [ ]:
import os
import boto3
import json
import random
from datetime import datetime
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import Swarm

import os 


# os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
# os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'

os.environ['AWS_REGION'] = 'us-east-1' 

bedrock = boto3.client('bedrock-runtime', region_name=region)

# Configure AWS credentials (ensure your AWS credentials are set up)
print("AWS Region:", os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'))
print("Setup complete!")

AWS Region: us-east-1
Setup complete!


## 2. Creating Tools for Our Agents

Let's create some tools that our agents can use to perform specific tasks:

In [2]:
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city.
    Args:
        city (str): The city name to get weather for.
    Returns:
        str: Weather information for the city.
    """
    conditions = ["sunny", "cloudy", "rainy", "snowy", "foggy"]
    temp = random.randint(-10, 35)
    condition = random.choice(conditions)
    return f"Weather in {city}: {condition}, {temp}°C"

@tool
def search_web(query: str) -> str:
    """Search the web for information.
    Args:
        query (str): The search query.
    Returns:
        str: Search results.
    """
    # Simulated web search results
    results = [
        f"Found information about {query} from TechCrunch: Latest developments in {query}...",
        f"Wikipedia entry for {query}: {query} is a technology that...",
        f"Research paper on {query}: Recent advances in {query} show promising results..."
    ]
    return "\n".join(results)

@tool
def calculate(expression: str) -> str:
    """Perform mathematical calculations.
    Args:
        expression (str): Mathematical expression to evaluate.
    Returns:
        str: Calculation result.
    """
    try:
        # Simple evaluation for basic math (in production, use a safer approach)
        result = eval(expression.replace('^', '**'))
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error calculating {expression}: {str(e)}"

@tool
def save_content(filename: str, content: str) -> str:
    """Save content to a file.
    Args:
        filename (str): Name of the file to save.
        content (str): Content to save.
    Returns:
        str: Confirmation message.
    """
    try:
        with open(filename, 'w') as f:
            f.write(content)
        return f"Content saved to {filename} successfully."
    except Exception as e:
        return f"Error saving to {filename}: {str(e)}"

print("Tools created successfully!")

Tools created successfully!


## 3. Creating Specialized Agents

Now let's create specialized agents with different roles and capabilities:

In [3]:
# Initialize Bedrock model
model = BedrockModel(model_id="us.amazon.nova-pro-v1:0")

# Research Agent - Specializes in gathering information
research_agent = Agent(
    model=model,
    name="research_agent",
    system_prompt="""You are a Research Agent specializing in gathering and analyzing information.
Your role in the swarm is to:
- Conduct thorough research on topics using available tools
- Provide factual, well-sourced information
- Identify key aspects and trends in the data
- Verify information accuracy before sharing

When working with other agents:
- Share your research findings clearly and concisely
- Hand off to creative or analytical agents when raw data needs processing
- Always cite your sources and methodology""",
    tools=[search_web, get_weather]
)

# Creative Agent - Specializes in content creation and innovation
creative_agent = Agent(
    model=model,
    name="creative_agent",
    system_prompt="""You are a Creative Agent specializing in content creation and innovative solutions.
Your role in the swarm is to:
- Transform research data into engaging content
- Generate creative ideas and approaches
- Write compelling narratives and presentations
- Think outside the box for unique solutions

When working with other agents:
- Build upon research findings to create original content
- Collaborate with analytical agents to ensure accuracy
- Hand off to quality assurance for review and refinement""",
    tools=[save_content]
)

# Analytical Agent - Specializes in data analysis and calculations
analytical_agent = Agent(
    model=model,
    name="analytical_agent",
    system_prompt="""You are an Analytical Agent specializing in data analysis and mathematical computations.
Your role in the swarm is to:
- Perform complex calculations and statistical analysis
- Identify patterns and trends in data
- Provide quantitative insights and metrics
- Validate numerical claims and projections

When working with other agents:
- Process raw data from research agents
- Provide analytical support for creative content
- Ensure mathematical accuracy in all outputs""",
    tools=[calculate]
)

# Quality Assurance Agent - Specializes in review and refinement
qa_agent = Agent(
    model=model,
    name="qa_agent",
    system_prompt="""You are a Quality Assurance Agent specializing in review and refinement.
Your role in the swarm is to:
- Review all work produced by other agents
- Identify errors, inconsistencies, or areas for improvement
- Ensure high quality standards are met
- Provide constructive feedback and suggestions

When working with other agents:
- Carefully examine all outputs for accuracy and quality
- Suggest improvements while maintaining the original intent
- Coordinate final deliverables and summaries
- Only complete the swarm task when quality standards are met""",
    tools=[save_content]
)

print("Specialized agents created successfully!")
print(f"Research Agent: {research_agent.name}")
print(f"Creative Agent: {creative_agent.name}")
print(f"Analytical Agent: {analytical_agent.name}")
print(f"QA Agent: {qa_agent.name}")

Specialized agents created successfully!
Research Agent: research_agent
Creative Agent: creative_agent
Analytical Agent: analytical_agent
QA Agent: qa_agent


## 4. Creating and Configuring the Swarm

Now let's create our multi-agent swarm using the Strands SDK:

In [4]:
# Create the swarm with our specialized agents
swarm = Swarm(
    [research_agent, creative_agent, analytical_agent, qa_agent],
    max_handoffs=15,  # Maximum number of agent handoffs
    max_iterations=20,  # Maximum total iterations
    execution_timeout=600.0,  # 10 minutes total timeout
    node_timeout=120.0,  # 2 minutes per agent
    repetitive_handoff_detection_window=6,  # Check last 6 handoffs for ping-pong
    repetitive_handoff_min_unique_agents=3  # Require at least 3 unique agents
)

print("Multi-agent swarm created successfully!")

Multi-agent swarm created successfully!


## 5. Testing the Swarm with Simple Tasks

Let's start with a simple task to see how our agents collaborate:

In [5]:
# Simple task: Weather analysis and report
simple_task = """Create a weather report for New York and London, 
then write a brief travel recommendation based on the weather conditions."""

print("Executing simple task...")
print(f"Task: {simple_task}")
print("\n" + "="*50 + "\n")

# Execute the swarm
result = swarm(simple_task)

print(f"\nTask Status: {result.status}")
print(f"Total Iterations: {result.execution_count}")
print(f"Execution Time: {result.execution_time:.2f}ms")
print(f"Agents Involved: {len(result.results)}")

Executing simple task...
Task: Create a weather report for New York and London, 
then write a brief travel recommendation based on the weather conditions.


<thinking> To fulfill the user's request, I need to gather current weather information for New York and London. Once I have this data, I will hand it off to the creative_agent to write a brief travel recommendation based on the weather conditions. </thinking>

Tool #1: get_weather

Tool #2: get_weather
<thinking> There seems to be an error in the temperature data for London, as 32°C is not typical for snowy weather. I will verify the weather conditions again to ensure accuracy. </thinking> 
Tool #3: get_weather
<thinking> The weather data for London has been corrected. Now I have accurate weather information for both New York and London. I will hand off this data to the creative_agent to write a brief travel recommendation. </thinking> 
Tool #4: handoff_to_agent
The task has been handed off to the creative_agent to create a brief t

In [25]:
# Display the agent collaboration flow
print("\nAgent Collaboration Flow:")
print("-" * 30)
for i, node in enumerate(result.node_history, 1):
    print(f"{i}. {node.node_id}")

# Show results from each agent
print("\nResults from Each Agent:")
print("=" * 40)
for agent_name, agent_result in result.results.items():
    print(f"\n{agent_name.upper()}:")
    print("-" * len(agent_name))
    print(agent_result.result)


Agent Collaboration Flow:
------------------------------
1. research_agent
2. creative_agent
3. qa_agent

Results from Each Agent:

RESEARCH_AGENT:
--------------
<thinking> The task has been handed off to the `creative_agent` to write a brief travel recommendation based on the weather conditions in New York and London. The swarm will consider the task complete once the `creative_agent` provides the recommendation. </thinking>

The task has been handed off to the `creative_agent` to write a brief travel recommendation based on the weather conditions in New York and London. Please wait for the `creative_agent` to provide the recommendation.


CREATIVE_AGENT:
--------------
The weather report has been saved to "weather_report.txt" successfully. Additionally, I have handed off the travel recommendation to the qa_agent for review and refinement. The qa_agent will ensure the recommendation is accurate and well-polished.

Here is the initial travel recommendation for your reference:
"Given 

## 6. Complex Multi-Agent Task

Now let's test our swarm with a more complex task that requires multiple types of expertise:

In [26]:
# Complex task: Market analysis and business plan
complex_task = """Create a comprehensive analysis of the AI chatbot market including:
1. Current market size and growth projections
2. Key competitors and their market share
3. Technology trends and innovations
4. A business plan for a new AI chatbot startup
5. Financial projections for the first 3 years
6. Risk analysis and mitigation strategies

Save the final report as 'ai_chatbot_market_analysis.txt'"""

print("Executing complex multi-agent task...")
print(f"Task: {complex_task}")
print("\n" + "="*60 + "\n")

# Execute the complex task
complex_result = swarm(complex_task)

print(f"\nTask Status: {complex_result.status}")
print(f"Total Iterations: {complex_result.execution_count}")
print(f"Execution Time: {complex_result.execution_time:.2f}ms")
print(f"Total Token Usage: {complex_result.accumulated_usage}")

Executing complex multi-agent task...
Task: Create a comprehensive analysis of the AI chatbot market including:
1. Current market size and growth projections
2. Key competitors and their market share
3. Technology trends and innovations
4. A business plan for a new AI chatbot startup
5. Financial projections for the first 3 years
6. Risk analysis and mitigation strategies

Save the final report as 'ai_chatbot_market_analysis.txt'


<thinking> To create a comprehensive analysis of the AI chatbot market, I need to gather detailed information on each of the requested aspects. I will start by searching for current market size and growth projections. Once I have this data, I will proceed to gather information on key competitors and their market share, technology trends and innovations, and then hand off specific tasks to other agents for the business plan, financial projections, and risk analysis. </thinking>


Tool #4: search_web
<thinking> The search results provide some sources for curre

In [30]:
# Analyze the complex task execution
print("\nComplex Task Agent Flow:")
print("-" * 35)
for i, node in enumerate(complex_result.node_history, 1):
    print(f"{i}. {node.node_id}")

# Show detailed results
print("\nDetailed Agent Contributions:")
print("=" * 45)
for agent_name, agent_result in complex_result.results.items():
    print(f"\n{agent_name.upper()} CONTRIBUTION:")
    print("-" * (len(agent_name) + 13))
    result_text = str(agent_result.result)
    if len(result_text) > 500:
        print(result_text[:500] + "\n... [truncated] ...")
    else:
        print(result_text)
    print(f"\nToken Usage: {agent_result.accumulated_usage}")


Complex Task Agent Flow:
-----------------------------------
1. research_agent
2. qa_agent

Detailed Agent Contributions:

RESEARCH_AGENT CONTRIBUTION:
---------------------------
<thinking> All tasks have been handed off to the appropriate agents. I will now wait for their responses to compile the final report. </thinking> 

<thinking> I will now summarize the information gathered and the tasks handed off to create the final report. </thinking>

**AI Chatbot Market Analysis**

1. **Current Market Size and Growth Projections:**
   - The AI chatbot market is rapidly growing, with significant investments and adoption across various industries.
   - According to TechCrunch, 
... [truncated] ...

Token Usage: {'inputTokens': 14301, 'outputTokens': 1479, 'totalTokens': 15780}

QA_AGENT CONTRIBUTION:
---------------------
The comprehensive analysis of the AI chatbot market has been successfully saved as 'ai_chatbot_market_analysis.txt'. 

Here is a summary of the content included in the rep

## 9. Best Practices and Lessons Learned

Let's summarize the best practices for multi-agent swarm implementation:

In [34]:
def print_best_practices():
    """Print best practices for multi-agent swarm development."""
    
    practices = {
        "🎯 Agent Specialization": [
            "Create agents with clear, distinct roles and expertise",
            "Avoid overlapping capabilities that might cause confusion",
            "Give each agent specific tools relevant to their role",
            "Write detailed system prompts that define agent behavior"
        ],
        "🔄 Coordination Patterns": [
            "Set appropriate timeout values for complex tasks",
            "Use repetitive handoff detection to prevent infinite loops",
            "Monitor agent handoff patterns for optimization opportunities",
            "Design clear handoff criteria in agent prompts"
        ],
        "⚡ Performance Optimization": [
            "Balance max_handoffs vs task complexity",
            "Monitor token usage across agents",
            "Use appropriate model sizes for different agent roles",
            "Implement caching for repeated operations"
        ],
        "🛡️ Error Handling": [
            "Implement graceful degradation for agent failures",
            "Set reasonable timeout values to prevent hanging",
            "Log agent interactions for debugging",
            "Have fallback strategies for critical tasks"
        ],
        "📊 Monitoring & Analytics": [
            "Track success rates and performance metrics",
            "Monitor agent utilization patterns",
            "Analyze handoff sequences for optimization",
            "Measure task completion times and costs"
        ]
    }
    
    print("🏆 MULTI-AGENT SWARM BEST PRACTICES")
    print("=" * 45)
    
    for category, items in practices.items():
        print(f"\n{category}")
        print("-" * len(category))
        for item in items:
            print(f"  • {item}")

print_best_practices()

🏆 MULTI-AGENT SWARM BEST PRACTICES

🎯 Agent Specialization
----------------------
  • Create agents with clear, distinct roles and expertise
  • Avoid overlapping capabilities that might cause confusion
  • Give each agent specific tools relevant to their role
  • Write detailed system prompts that define agent behavior

🔄 Coordination Patterns
-----------------------
  • Set appropriate timeout values for complex tasks
  • Use repetitive handoff detection to prevent infinite loops
  • Monitor agent handoff patterns for optimization opportunities
  • Design clear handoff criteria in agent prompts

⚡ Performance Optimization
--------------------------
  • Balance max_handoffs vs task complexity
  • Monitor token usage across agents
  • Use appropriate model sizes for different agent roles
  • Implement caching for repeated operations

🛡️ Error Handling
-----------------
  • Implement graceful degradation for agent failures
  • Set reasonable timeout values to prevent hanging
  • Log age

## 10. Conclusion and Next Steps

Congratulations! You've successfully implemented a multi-agent swarm system using Amazon Bedrock and the Strands SDK. Let's summarize what we've accomplished:

In [35]:
def print_lab_summary():
    """Print a summary of what was accomplished in this lab."""
    
    print("🎉 LAB 8 COMPLETION SUMMARY")
    print("=" * 30)
    
    accomplishments = [
        "✅ Created specialized AI agents with distinct roles and capabilities",
        "✅ Implemented multi-agent coordination using Strands SDK Swarm pattern",
        "✅ Integrated Amazon Bedrock Nova Pro model for agent intelligence",
        "✅ Built custom tools for weather, search, calculations, and file operations",
        "✅ Demonstrated autonomous agent handoff and collaboration",
        "✅ Created monitoring and orchestration systems for swarm management",
        "✅ Analyzed performance metrics and optimization strategies",
        "✅ Explored advanced patterns for specialized swarm configurations"
    ]
    
    print("\n🏆 Key Accomplishments:")
    for item in accomplishments:
        print(f"  {item}")
    
    next_steps = [
        "🚀 Experiment with different agent configurations and roles",
        "🔧 Integrate real-world APIs and data sources",
        "📊 Implement advanced monitoring and logging systems",
        "🌐 Deploy swarms to production environments",
        "🤖 Explore other Bedrock models for different agent capabilities",
        "🔄 Implement persistent memory and learning capabilities",
        "📈 Scale to larger swarms with more specialized agents"
    ]
    
    print("\n🎯 Suggested Next Steps:")
    for item in next_steps:
        print(f"  {item}")
    
    print("\n💡 Key Concepts Learned:")
    concepts = [
        "Multi-agent system architecture and design patterns",
        "Agent specialization and role-based coordination",
        "Autonomous handoff mechanisms and shared context",
        "Performance monitoring and optimization techniques",
        "Error handling and timeout management in distributed systems",
        "Integration of cloud AI services with multi-agent frameworks"
    ]
    
    for concept in concepts:
        print(f"  📚 {concept}")

print_lab_summary()

🎉 LAB 8 COMPLETION SUMMARY

🏆 Key Accomplishments:
  ✅ Created specialized AI agents with distinct roles and capabilities
  ✅ Implemented multi-agent coordination using Strands SDK Swarm pattern
  ✅ Integrated Amazon Bedrock Nova Pro model for agent intelligence
  ✅ Built custom tools for weather, search, calculations, and file operations
  ✅ Demonstrated autonomous agent handoff and collaboration
  ✅ Created monitoring and orchestration systems for swarm management
  ✅ Analyzed performance metrics and optimization strategies
  ✅ Explored advanced patterns for specialized swarm configurations

🎯 Suggested Next Steps:
  🚀 Experiment with different agent configurations and roles
  🔧 Integrate real-world APIs and data sources
  📊 Implement advanced monitoring and logging systems
  🌐 Deploy swarms to production environments
  🤖 Explore other Bedrock models for different agent capabilities
  🔄 Implement persistent memory and learning capabilities
  📈 Scale to larger swarms with more special

In [36]:
print("\n🎓 Lab 8 completed successfully!")
print("You now have hands-on experience with multi-agent swarm systems!")


🎓 Lab 8 completed successfully!
You now have hands-on experience with multi-agent swarm systems!
